# Multimodal Fake News Detection using Text–Image Fusion
**Dataset:** Fakeddit (Reddit posts: headline + image), 2-way labels — `0 = fake`, `1 = real`

| # | Objective (from the slides) | Section |
|---|---|---|
| 1 | Cleaned, balanced text–image subset of Fakeddit | §2 |
| 2 | Text-only and image-only baselines on frozen features | §3, §5 |
| 3 | Early, late and attention fusion compared on the same test split, accuracy + macro-F1, averaged over seeds | §5, §6 |
| 4 | Modality-contribution explanation by ablation | §7 |
| 5 | Deploy the best model as a Streamlit web app | §8 |

Also included: §4 a classical **TF-IDF baseline** (same pipeline as the reference repo `kapilsinghnegi/Fake-News-Detection`) run on the Fakeddit headlines, so you can show how much the deep / multimodal models add.

**Runtime:** Colab → *Runtime ▸ Change runtime type ▸ T4 GPU*. Every expensive step (images, features) is cached to Google Drive, so re-running is fast.

## 0 · Setup

In [ ]:
import os, sys, re, string, json, time, random, warnings, shutil
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")

# ---------------- configuration ----------------
ROOT = r'C:\Users\user\Desktop\fakeddit_project'

DATA_DIR, IMG_DIR, FEAT_DIR = f'{ROOT}/data', f'{ROOT}/images', f'{ROOT}/features'
OUT_DIR, MODEL_DIR = f'{ROOT}/outputs', f'{ROOT}/models'
for d in [ROOT, DATA_DIR, IMG_DIR, FEAT_DIR, OUT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

N_TRAIN_PER_CLASS = 3000     # 6,000 train+val posts (balanced)
N_TEST_PER_CLASS  = 750      # 1,500 held-out test posts
VAL_FRAC   = 0.15            # 5,100 train / 900 validation
MAX_LEN    = 64              # DistilBERT tokens
IMG_SIZE   = 224
TEXT_POOL  = 'mean'          # 'mean' (recommended for frozen BERT) or 'cls'
SPLIT_SEED = 42              # data split is fixed; only model seeds vary
SEEDS      = [0, 1, 2, 3, 4] # results are averaged over these
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| project folder:', ROOT)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

### Shared code module
`fakenews_core.py` holds the feature extractors, all five models and the explanation function. It's written to the project folder so **the notebook and the Streamlit app use exactly the same code**.

In [ ]:
%%writefile {ROOT}/fakenews_core.py
"""
fakenews_core.py
Shared code for the notebook AND the Streamlit app, so both use exactly the
same feature extractors and model definitions.

Multimodal Fake News Detection using Text-Image Fusion (Fakeddit, 2-way labels)
Label convention: 0 = fake, 1 = real
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

TEXT_DIM, IMAGE_DIM, NUM_CLASSES = 768, 2048, 2
LABEL_NAMES = {0: "Fake", 1: "Real"}
TEXT_MODEL_NAME = "distilbert-base-uncased"


# ---------------------------------------------------------------------------
# 1. Frozen feature extractors (DistilBERT 768-d, ResNet50 2048-d)
# ---------------------------------------------------------------------------
class FeatureExtractor:
    """Frozen DistilBERT + ResNet50. Heavy imports happen lazily."""

    def __init__(self, device="cpu", max_len=64, text_pool="mean", img_size=224):
        from transformers import AutoTokenizer, AutoModel
        import torchvision
        from torchvision import transforms

        self.device, self.max_len, self.text_pool = device, max_len, text_pool
        self.tok = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
        self.bert = AutoModel.from_pretrained(TEXT_MODEL_NAME).eval().to(device)

        weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
        cnn = torchvision.models.resnet50(weights=weights)
        cnn.fc = nn.Identity()  # keep the 2048-d pooled features
        self.cnn = cnn.eval().to(device)
        self.tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    @torch.no_grad()
    def text(self, texts, batch_size=64):
        out = []
        for s in range(0, len(texts), batch_size):
            enc = self.tok(list(texts[s:s + batch_size]), padding=True, truncation=True,
                           max_length=self.max_len, return_tensors="pt").to(self.device)
            h = self.bert(**enc).last_hidden_state                     # (B, L, 768)
            if self.text_pool == "cls":
                v = h[:, 0]
            else:                                                       # masked mean pooling
                m = enc["attention_mask"].unsqueeze(-1).float()
                v = (h * m).sum(1) / m.sum(1).clamp(min=1e-6)
            out.append(v.float().cpu().numpy())
        return np.concatenate(out).astype(np.float32)

    @torch.no_grad()
    def image(self, pil_images, batch_size=64):
        out = []
        for s in range(0, len(pil_images), batch_size):
            x = torch.stack([self.tf(im.convert("RGB")) for im in pil_images[s:s + batch_size]])
            out.append(self.cnn(x.to(self.device)).float().cpu().numpy())
        return np.concatenate(out).astype(np.float32)


# ---------------------------------------------------------------------------
# 2. Classifiers on frozen features. Every model: forward(t, i) -> (logits, info)
#    "Masking" a modality = feeding zeros. Features are standardised with
#    train-set mean/std, so zeros == the average train example (mean imputation).
# ---------------------------------------------------------------------------
def _head(in_dim, hidden, dropout):
    return nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(dropout), nn.Linear(in_dim, hidden),
                         nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, NUM_CLASSES))


class TextOnly(nn.Module):
    uses = ("text",)

    def __init__(self, hidden=256, dropout=0.3, **_):
        super().__init__()
        self.head = _head(TEXT_DIM, hidden, dropout)

    def forward(self, t, i):
        return self.head(t), {}


class ImageOnly(nn.Module):
    uses = ("image",)

    def __init__(self, hidden=256, dropout=0.3, **_):
        super().__init__()
        self.head = _head(IMAGE_DIM, hidden, dropout)

    def forward(self, t, i):
        return self.head(i), {}


class EarlyFusion(nn.Module):
    """Feature-level fusion: concatenate [text ; image] -> MLP."""
    uses = ("text", "image")

    def __init__(self, hidden=512, dropout=0.3, **_):
        super().__init__()
        self.head = _head(TEXT_DIM + IMAGE_DIM, hidden, dropout)

    def forward(self, t, i):
        return self.head(torch.cat([t, i], dim=1)), {}


class LateFusion(nn.Module):
    """Decision-level fusion: separate text and image classifiers whose logits are
    combined with a learned weight alpha. Each branch also gets its own auxiliary
    loss (see aux_loss) so that both stay independently meaningful."""
    uses = ("text", "image")

    def __init__(self, hidden=256, dropout=0.3, **_):
        super().__init__()
        self.text_head = _head(TEXT_DIM, hidden, dropout)
        self.image_head = _head(IMAGE_DIM, hidden, dropout)
        self.w = nn.Parameter(torch.zeros(1))  # alpha = sigmoid(w), starts at 0.5

    def forward(self, t, i):
        lt, li = self.text_head(t), self.image_head(i)
        a = torch.sigmoid(self.w)
        return a * lt + (1 - a) * li, {"logits_text": lt, "logits_image": li, "alpha": a}

    @staticmethod
    def aux_loss(info, y, crit):
        return 0.5 * (crit(info["logits_text"], y) + crit(info["logits_image"], y))


class AttentionFusion(nn.Module):
    """Attention-based fusion.
    1) project text and image to a shared d-dim space -> 2 modality tokens
    2) multi-head self-attention between the tokens (cross-modal interaction)
    3) a learned gate scores each token; softmax -> per-sample modality weights
    4) weighted sum -> classifier.  The gate weights are returned for explanation."""
    uses = ("text", "image")

    def __init__(self, d=256, heads=4, dropout=0.3, **_):
        super().__init__()
        self.pt = nn.Sequential(nn.LayerNorm(TEXT_DIM), nn.Dropout(dropout), nn.Linear(TEXT_DIM, d))
        self.pi = nn.Sequential(nn.LayerNorm(IMAGE_DIM), nn.Dropout(dropout), nn.Linear(IMAGE_DIM, d))
        self.mod_emb = nn.Parameter(torch.randn(2, d) * 0.02)   # tells tokens apart
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * d, d))
        self.gate = nn.Linear(d, 1)
        self.cls = nn.Sequential(nn.Dropout(dropout), nn.Linear(d, NUM_CLASSES))

    def forward(self, t, i):
        tok = torch.stack([self.pt(t), self.pi(i)], dim=1) + self.mod_emb      # (B, 2, d)
        a, attn_w = self.attn(tok, tok, tok, need_weights=True)                 # attn_w (B, 2, 2)
        z = self.ln1(tok + a)
        z = self.ln2(z + self.ffn(z))
        g = torch.softmax(self.gate(z).squeeze(-1), dim=1)                      # (B, 2)
        fused = (g.unsqueeze(-1) * z).sum(1)
        return self.cls(fused), {"gate": g, "attn": attn_w}


MODELS = {
    "text_only": TextOnly,
    "image_only": ImageOnly,
    "early_fusion": EarlyFusion,
    "late_fusion": LateFusion,
    "attention_fusion": AttentionFusion,
}


def build_model(name, **kw):
    return MODELS[name](**kw)


# ---------------------------------------------------------------------------
# 3. Inference + modality-contribution explanation (ablation)
# ---------------------------------------------------------------------------
@torch.no_grad()
def predict_proba(model, t, i, mask=None):
    """t, i: standardised float tensors. mask: None | 'text' | 'image' (zeroed)."""
    model.eval()
    if mask == "text":
        t = torch.zeros_like(t)
    if mask == "image":
        i = torch.zeros_like(i)
    logits, info = model(t, i)
    return F.softmax(logits, dim=1), info


@torch.no_grad()
def explain(model, t, i):
    """Per-sample modality contribution by ablation.
    contribution(text)  = |p_full(pred) - p_text_masked(pred)|
    contribution(image) = |p_full(pred) - p_image_masked(pred)|
    Shares are these normalised to sum to 1."""
    p_full, info = predict_proba(model, t, i)
    pred = p_full.argmax(1)
    idx = torch.arange(len(pred), device=pred.device)
    p_nt, _ = predict_proba(model, t, i, mask="text")
    p_ni, _ = predict_proba(model, t, i, mask="image")
    c_text = (p_full[idx, pred] - p_nt[idx, pred]).abs()
    c_img = (p_full[idx, pred] - p_ni[idx, pred]).abs()
    tot = (c_text + c_img).clamp(min=1e-8)
    out = {
        "pred": pred.cpu().numpy(),
        "prob": p_full.cpu().numpy(),
        "text_share": (c_text / tot).cpu().numpy(),
        "image_share": (c_img / tot).cpu().numpy(),
        "prob_text_only_view": p_ni.cpu().numpy(),   # image masked -> the text's view
        "prob_image_only_view": p_nt.cpu().numpy(),  # text masked  -> the image's view
    }
    if "gate" in info:
        out["gate_text"], out["gate_image"] = info["gate"][:, 0].cpu().numpy(), info["gate"][:, 1].cpu().numpy()
    if "alpha" in info:
        out["alpha_text"] = float(info["alpha"].item())
    return out


def standardise(x, mean, std):
    return (x - mean) / std

In [ ]:
sys.path.insert(0, ROOT)
import importlib, fakenews_core as core
importlib.reload(core)
from fakenews_core import build_model, explain, predict_proba, LABEL_NAMES, MODELS
print('Models available:', list(MODELS))

## 1 · Get the Fakeddit files
Download from the official Fakeddit page (github.com/entitize/Fakeddit → *multimodal only samples*) and put these two files in `MyDrive/fakeddit_project/data/`:
- `multimodal_train.tsv`
- `multimodal_test_public.tsv`

**Already built your 6,000/1,500 subset?** Put it in `data/` as `subset_trainval.csv` and `subset_test.csv` (columns `id, clean_title, image_url, label`) with the images in `images/<id>.jpg`, and §2 will skip straight to using it.

In [ ]:
TRAIN_TSV = f'{DATA_DIR}/multimodal_train.tsv'
TEST_TSV  = f'{DATA_DIR}/multimodal_test_public.tsv'
SUB_TRV, SUB_TEST = f'{DATA_DIR}/subset_trainval.csv', f'{DATA_DIR}/subset_test.csv'
have_subset = os.path.exists(SUB_TRV) and os.path.exists(SUB_TEST)
print('Existing subset found:', have_subset)
if not have_subset:
    for p in [TRAIN_TSV, TEST_TSV]:
        assert os.path.exists(p), f'Missing {p} — download it from the Fakeddit page (see above).'

## 2 · Build the cleaned, balanced text–image subset  *(Objective 1)*
Filters: has an image, image hosted on `i.redd.it` / `imgur` (most stable links), non-empty title with ≥3 words, duplicates removed. We over-sample candidates because some image links are dead, then keep the first *N* per class that download correctly, resize them to 224×224 and save them as JPEG.

In [ ]:
def load_fakeddit(path):
    cols = ['id', 'clean_title', 'image_url', 'hasImage', '2_way_label']
    df = pd.read_csv(path, sep='\t', usecols=cols, low_memory=False)
    df = df.dropna(subset=['clean_title', 'image_url', '2_way_label'])
    df = df[df['hasImage'].astype(str).str.lower() == 'true']
    df = df[df['image_url'].str.contains(r'i\.redd\.it|imgur\.com', regex=True)]
    df['clean_title'] = df['clean_title'].astype(str).str.strip()
    df = df[df['clean_title'].str.split().str.len() >= 3]
    df = df.drop_duplicates('id').drop_duplicates('clean_title')
    df['label'] = df['2_way_label'].astype(int)          # 0 = fake, 1 = real
    return df[['id', 'clean_title', 'image_url', 'label']].reset_index(drop=True)

def fix_url(u):
    # imgur page links (imgur.com/abc) -> direct image (i.imgur.com/abc.jpg)
    m = re.match(r'https?://(?:www\.|m\.)?imgur\.com/(\w+)$', u)
    return f'https://i.imgur.com/{m.group(1)}.jpg' if m else u

def download_one(row):
    import requests
    from PIL import Image
    path = f'{IMG_DIR}/{row.id}.jpg'
    if os.path.exists(path):
        return row.id, True
    try:
        r = requests.get(fix_url(row.image_url), timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        if r.status_code != 200 or 'removed' in r.url:      # imgur "removed" placeholder
            return row.id, False
        im = Image.open(BytesIO(r.content)).convert('RGB')
        if min(im.size) < 64:                               # placeholders / icons
            return row.id, False
        im.resize((IMG_SIZE, IMG_SIZE)).save(path, 'JPEG', quality=90)
        return row.id, True
    except Exception:
        return row.id, False

def sample_with_images(df, n_per_class, seed, oversample=1.6, workers=32):
    picked = []
    for lab in [0, 1]:
        pool = df[df.label == lab].sample(frac=1, random_state=seed)
        got, start = [], 0
        while len(got) < n_per_class and start < len(pool):
            chunk = pool.iloc[start:start + int((n_per_class - len(got)) * oversample) + 50]
            start += len(chunk)
            with ThreadPoolExecutor(workers) as ex:
                ok = dict(ex.map(download_one, chunk.itertuples()))
            got += [r for r in chunk.itertuples() if ok[r.id]]
            print(f'  label {lab}: {min(len(got), n_per_class)}/{n_per_class} images ok')
        picked.append(pd.DataFrame(got[:n_per_class]).drop(columns='Index'))
    return pd.concat(picked).sample(frac=1, random_state=seed).reset_index(drop=True)

if not have_subset:
    t0 = time.time()
    full_train, full_test = load_fakeddit(TRAIN_TSV), load_fakeddit(TEST_TSV)
    print('After filtering:', len(full_train), 'train /', len(full_test), 'test candidates')
    trainval = sample_with_images(full_train, N_TRAIN_PER_CLASS, SPLIT_SEED)
    test     = sample_with_images(full_test,  N_TEST_PER_CLASS,  SPLIT_SEED)
    trainval.to_csv(SUB_TRV, index=False); test.to_csv(SUB_TEST, index=False)
    print(f'Done in {(time.time()-t0)/60:.1f} min')

trainval, test = pd.read_csv(SUB_TRV), pd.read_csv(SUB_TEST)
# keep only rows whose image exists on disk
trainval = trainval[[os.path.exists(f'{IMG_DIR}/{i}.jpg') for i in trainval.id]].reset_index(drop=True)
test     = test[[os.path.exists(f'{IMG_DIR}/{i}.jpg') for i in test.id]].reset_index(drop=True)

train_df, val_df = train_test_split(trainval, test_size=VAL_FRAC, stratify=trainval.label,
                                    random_state=SPLIT_SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)
for name, d in [('train', train_df), ('val', val_df), ('test', test)]:
    print(f'{name:5s}: {len(d):5d} posts | fake={int((d.label==0).sum())} real={int((d.label==1).sum())}')

In [ ]:
# Sanity check: look at a few posts per class (confirms 0 = fake, 1 = real)
from PIL import Image
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for r, lab in enumerate([0, 1]):
    for c, row in enumerate(train_df[train_df.label == lab].head(4).itertuples()):
        axes[r, c].imshow(Image.open(f'{IMG_DIR}/{row.id}.jpg'))
        axes[r, c].set_title(f'[{LABEL_NAMES[lab]}] {row.clean_title[:45]}', fontsize=9)
        axes[r, c].axis('off')
plt.tight_layout(); plt.show()

## 3 · Extract frozen features  *(Objective 2)*
- **Text:** DistilBERT (`distilbert-base-uncased`), max 64 tokens → 768-d (mean-pooled)
- **Image:** ResNet50 (ImageNet weights, classifier removed) → 2048-d

Features are cached as `.npy` on Drive (≈2–4 min on a T4, only once).

In [ ]:
def feats_path(split, kind): return f'{FEAT_DIR}/{split}_{kind}_{TEXT_POOL}.npy'
splits = {'train': train_df, 'val': val_df, 'test': test}

need = any(not os.path.exists(feats_path(s, k)) for s in splits for k in ['text', 'image', 'ids'])
if need:
    from PIL import Image
    fx = core.FeatureExtractor(device=DEVICE, max_len=MAX_LEN, text_pool=TEXT_POOL, img_size=IMG_SIZE)
    for s, d in splits.items():
        t0 = time.time()
        np.save(feats_path(s, 'text'), fx.text(d.clean_title.tolist()))
        imgs = [Image.open(f'{IMG_DIR}/{i}.jpg') for i in d.id]
        np.save(feats_path(s, 'image'), fx.image(imgs))
        np.save(feats_path(s, 'ids'), d.id.values.astype(str))
        print(f'{s}: {len(d)} posts in {time.time()-t0:.0f}s')
    del fx; torch.cuda.empty_cache()

F_ = {s: {k: np.load(feats_path(s, k), allow_pickle=True) for k in ['text', 'image', 'ids']} for s in splits}
for s, d in splits.items():   # make sure features line up with the dataframe rows
    assert (F_[s]['ids'] == d.id.values.astype(str)).all(), f'{s}: id order mismatch - delete {FEAT_DIR} and re-run'
    F_[s]['y'] = d.label.values.astype(np.int64)

# Standardise with TRAIN statistics only (zero = 'average train post', used for masking)
stats = {k: (F_['train'][k].mean(0), F_['train'][k].std(0) + 1e-6) for k in ['text', 'image']}
for s in splits:
    for k in ['text', 'image']:
        F_[s][k + '_z'] = ((F_[s][k] - stats[k][0]) / stats[k][1]).astype(np.float32)
np.savez(f'{MODEL_DIR}/feature_stats.npz', text_mean=stats['text'][0], text_std=stats['text'][1],
         image_mean=stats['image'][0], image_std=stats['image'][1])
print({s: (F_[s]['text'].shape, F_[s]['image'].shape) for s in splits})

## 4 · Classical baseline — TF-IDF + ML (reference-repo pipeline, on Fakeddit titles)
Same steps as `kapilsinghnegi/Fake-News-Detection`: `wordopt` cleaning → TF-IDF → Logistic Regression, Decision Tree, Gradient Boosting, Random Forest. Trained on the same 5,100 training posts and tested on the same 1,500 test posts, so the numbers are directly comparable with §6.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

def wordopt(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(train_df.clean_title.apply(wordopt))
Xte = vec.transform(test.clean_title.apply(wordopt))
classical = {'TF-IDF + LogReg': LogisticRegression(max_iter=2000, C=2.0),
             'TF-IDF + DecisionTree': DecisionTreeClassifier(random_state=0),
             'TF-IDF + GradBoost': GradientBoostingClassifier(random_state=0),
             'TF-IDF + RandomForest': RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)}
classical_rows = []
for name, clf in classical.items():
    pred = clf.fit(Xtr, train_df.label).predict(Xte)
    classical_rows.append({'model': name, 'accuracy': accuracy_score(test.label, pred),
                           'macro_f1': f1_score(test.label, pred, average='macro')})
classical_res = pd.DataFrame(classical_rows)
classical_res.to_csv(f'{OUT_DIR}/classical_baselines.csv', index=False)
classical_res.round(4)

## 5 · Train the five models on frozen features  *(Objectives 2 & 3)*
| Model | Fusion type | Idea |
|---|---|---|
| `text_only` | — | MLP on DistilBERT features |
| `image_only` | — | MLP on ResNet50 features |
| `early_fusion` | feature level | concatenate [768 ; 2048] → MLP |
| `late_fusion` | decision level | separate text and image classifiers, logits mixed with a learned weight α (+ auxiliary loss per branch) |
| `attention_fusion` | attention | project both to 256-d tokens → multi-head attention between them → learned gate gives per-post modality weights → classifier |

Same recipe for all: AdamW, early stopping on **validation macro-F1** (patience 6), best epoch restored. Fusion models use *modality dropout* (10 % of the time one modality is blanked during training) so they can't lean on only one input. Each model is trained with all 5 seeds.

In [ ]:
HP = {  # hyper-parameters per model
    'text_only':        dict(hidden=256, dropout=0.3, lr=1e-3, wd=1e-2),
    'image_only':       dict(hidden=256, dropout=0.3, lr=1e-3, wd=1e-2),
    'early_fusion':     dict(hidden=512, dropout=0.4, lr=1e-3, wd=1e-2),
    'late_fusion':      dict(hidden=256, dropout=0.3, lr=1e-3, wd=1e-2),
    'attention_fusion': dict(d=256, heads=4, dropout=0.3, lr=5e-4, wd=1e-2),
}
EPOCHS, PATIENCE, BATCH, MOD_DROPOUT = 40, 6, 64, 0.10

def tensors(split):
    return (torch.tensor(F_[split]['text_z']), torch.tensor(F_[split]['image_z']), torch.tensor(F_[split]['y']))

T = {s: tensors(s) for s in splits}

@torch.no_grad()
def evaluate(model, split, mask=None):
    t, i, y = (x.to(DEVICE) for x in T[split])
    p, info = predict_proba(model, t, i, mask=mask)
    pred = p.argmax(1).cpu().numpy(); yy = y.cpu().numpy()
    return {'accuracy': accuracy_score(yy, pred), 'macro_f1': f1_score(yy, pred, average='macro'),
            'pred': pred, 'prob': p.cpu().numpy()}

def train_model(name, seed, verbose=False):
    set_seed(seed)
    hp = HP[name]
    model = build_model(name, **hp).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss()
    t_all, i_all, y_all = (x.to(DEVICE) for x in T['train'])
    n = len(y_all); g = torch.Generator(device='cpu').manual_seed(seed)
    is_fusion = len(model.uses) == 2
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(n, generator=g).to(DEVICE); tot = 0
        for s in range(0, n, BATCH):
            idx = perm[s:s + BATCH]; t, i, y = t_all[idx], i_all[idx], y_all[idx]
            if is_fusion and MOD_DROPOUT > 0:            # modality dropout
                r = torch.rand(len(y), device=DEVICE)
                t = t * (r >= MOD_DROPOUT / 2).float().unsqueeze(1)
                i = i * ((r < MOD_DROPOUT / 2) | (r >= MOD_DROPOUT)).float().unsqueeze(1)
            logits, info = model(t, i)
            loss = crit(logits, y)
            if hasattr(model, 'aux_loss'):
                loss = loss + model.aux_loss(info, y, crit)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(y)
        sched.step()
        v = evaluate(model, 'val')
        hist.append({'epoch': ep + 1, 'train_loss': tot / n, 'val_acc': v['accuracy'], 'val_f1': v['macro_f1']})
        if v['macro_f1'] > best:
            best, bad = v['macro_f1'], 0
            best_state = {k: x.detach().clone() for k, x in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
        if verbose: print(hist[-1])
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

In [ ]:
results, runs = [], {}     # runs[(model, seed)] = (model, history, test-eval)
t0 = time.time()
for name in MODELS:
    for seed in SEEDS:
        model, hist = train_model(name, seed)
        te, va = evaluate(model, 'test'), evaluate(model, 'val')
        runs[(name, seed)] = (model, hist, te)
        results.append({'model': name, 'seed': seed, 'best_epoch': int(hist.val_f1.idxmax()) + 1,
                        'val_macro_f1': va['macro_f1'], 'test_accuracy': te['accuracy'], 'test_macro_f1': te['macro_f1']})
        print(f'{name:17s} seed {seed}: test acc {te["accuracy"]:.4f} | macro-F1 {te["macro_f1"]:.4f} '
              f'(best epoch {results[-1]["best_epoch"]})')
print(f'All runs: {(time.time()-t0)/60:.1f} min')
res = pd.DataFrame(results); res.to_csv(f'{OUT_DIR}/all_runs.csv', index=False)

## 6 · Results — same test split, mean ± std over 5 seeds  *(Objective 3)*

In [ ]:
summary = (res.groupby('model', sort=False)
              .agg(acc_mean=('test_accuracy', 'mean'), acc_std=('test_accuracy', 'std'),
                   f1_mean=('test_macro_f1', 'mean'), f1_std=('test_macro_f1', 'std'),
                   val_f1_mean=('val_macro_f1', 'mean'))
              .sort_values('f1_mean', ascending=False))
summary.to_csv(f'{OUT_DIR}/summary_mean_std.csv')
table = pd.DataFrame({'Accuracy': [f'{m:.4f} ± {s:.4f}' for m, s in zip(summary.acc_mean, summary.acc_std)],
                      'Macro-F1': [f'{m:.4f} ± {s:.4f}' for m, s in zip(summary.f1_mean, summary.f1_std)]},
                     index=summary.index)
print('Classical TF-IDF baselines (single run):'); display(classical_res.round(4))
print('Frozen-feature models (5 seeds):'); display(table)

# Model selection uses VALIDATION F1 (never the test set)
BEST_MODEL = summary.val_f1_mean.idxmax()
best_seed = res[res.model == BEST_MODEL].sort_values('val_macro_f1').iloc[-1].seed
print(f'Best model (by mean validation macro-F1): {BEST_MODEL} | saved seed: {best_seed}')

In [ ]:
# Bar chart with error bars
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
order = summary.index.tolist()
for a, (m, s, title) in zip(ax, [('acc_mean', 'acc_std', 'Test accuracy'), ('f1_mean', 'f1_std', 'Test macro-F1')]):
    colors = ['#4C72B0' if n in ('text_only', 'image_only') else '#DD8452' for n in order]
    a.bar(order, summary[m], yerr=summary[s], capsize=5, color=colors)
    a.set_title(title + ' (mean ± std, 5 seeds)'); a.set_ylim(max(0, summary[m].min() - 0.1), min(1, summary[m].max() + 0.05))
    a.tick_params(axis='x', rotation=25)
    for k, v in enumerate(summary[m]): a.text(k, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/results_bar.png', dpi=150); plt.show()

# Confusion matrices (seed 0 of each model)
fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 3.6))
for a, name in zip(axes, MODELS):
    cm = confusion_matrix(F_['test']['y'], runs[(name, SEEDS[0])][2]['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=a,
                xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    a.set_title(name); a.set_xlabel('Predicted'); a.set_ylabel('True')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/confusion_matrices.png', dpi=150); plt.show()

print(f'Classification report — {BEST_MODEL}, seed {best_seed}')
print(classification_report(F_['test']['y'], runs[(BEST_MODEL, best_seed)][2]['pred'], target_names=['Fake', 'Real'], digits=4))

In [ ]:
# Is fusion's gain over text-only consistent? Paired comparison per seed + McNemar test (seed 0)
from scipy.stats import binomtest
base = res[res.model == 'text_only'].set_index('seed').test_macro_f1
for name in ['early_fusion', 'late_fusion', 'attention_fusion']:
    diff = res[res.model == name].set_index('seed').test_macro_f1 - base
    a = runs[('text_only', SEEDS[0])][2]['pred'] == F_['test']['y']
    b = runs[(name, SEEDS[0])][2]['pred'] == F_['test']['y']
    n01, n10 = int((~a & b).sum()), int((a & ~b).sum())
    p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    print(f'{name:17s}: ΔF1 vs text-only = {diff.mean():+.4f} ± {diff.std():.4f}, better on {int((diff>0).sum())}/{len(diff)} seeds | '
          f'McNemar p = {p:.4f}')

# Training curves for the best model
hist = runs[(BEST_MODEL, best_seed)][1]
fig, a = plt.subplots(1, 2, figsize=(11, 3.5))
a[0].plot(hist.epoch, hist.train_loss); a[0].set_title('Train loss'); a[0].set_xlabel('epoch')
a[1].plot(hist.epoch, hist.val_acc, label='val acc'); a[1].plot(hist.epoch, hist.val_f1, label='val macro-F1')
a[1].legend(); a[1].set_title(f'{BEST_MODEL} — validation'); a[1].set_xlabel('epoch')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=150); plt.show()

## 7 · Modality-contribution explanation by ablation  *(Objective 4)*
Two levels:
1. **Global** — at test time, blank one modality (replace it with the average train post) and measure how much macro-F1 drops. Big drop ⇒ the model relies on that modality.
2. **Per post** — for each prediction, *text share* and *image share* = how much the predicted-class probability changes when that modality is removed (normalised to sum to 1). For `attention_fusion` we also compare with its learned gate weights.

In [ ]:
abl = []
for name in ['early_fusion', 'late_fusion', 'attention_fusion']:
    for seed in SEEDS:
        m = runs[(name, seed)][0]
        full = evaluate(m, 'test')['macro_f1']
        no_t = evaluate(m, 'test', mask='text')['macro_f1']
        no_i = evaluate(m, 'test', mask='image')['macro_f1']
        abl.append({'model': name, 'seed': seed, 'full': full, 'text_removed': no_t, 'image_removed': no_i,
                    'drop_without_text': full - no_t, 'drop_without_image': full - no_i})
abl = pd.DataFrame(abl); abl.to_csv(f'{OUT_DIR}/ablation_runs.csv', index=False)
abl_sum = abl.groupby('model', sort=False)[['full', 'text_removed', 'image_removed', 'drop_without_text', 'drop_without_image']].mean()
display(abl_sum.round(4))

ax = abl_sum[['drop_without_text', 'drop_without_image']].plot.bar(figsize=(8, 4), rot=0, color=['#4C72B0', '#55A868'])
ax.set_ylabel('Macro-F1 drop'); ax.set_title('Global ablation: performance lost when a modality is removed')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/ablation_global.png', dpi=150); plt.show()

In [ ]:
# Per-post contributions for the best fusion model
fusion_names = ['early_fusion', 'late_fusion', 'attention_fusion']
EXPL_MODEL = BEST_MODEL if BEST_MODEL in fusion_names else summary.loc[fusion_names].val_f1_mean.idxmax()
m = runs[(EXPL_MODEL, SEEDS[0])][0]
t, i, y = (x.to(DEVICE) for x in T['test'])
ex = explain(m, t, i)
expl = test[['id', 'clean_title', 'label']].copy()
expl['pred'] = ex['pred']; expl['p_real'] = ex['prob'][:, 1]
expl['text_share'] = ex['text_share']; expl['image_share'] = ex['image_share']
if 'gate_text' in ex: expl['gate_text'], expl['gate_image'] = ex['gate_text'], ex['gate_image']
expl['correct'] = expl.pred == expl.label
expl.to_csv(f'{OUT_DIR}/per_post_explanations_{EXPL_MODEL}.csv', index=False)

print(f'{EXPL_MODEL}: mean text share {expl.text_share.mean():.3f} | mean image share {expl.image_share.mean():.3f}')
display(expl.groupby(['label', 'correct'])[['text_share', 'image_share']].mean().rename(index={0: 'fake', 1: 'real'}).round(3))
if 'gate_text' in expl:
    print('Correlation (gate weight on text vs ablation text share):', round(expl[['gate_text', 'text_share']].corr().iloc[0, 1], 3))

fig, a = plt.subplots(1, 2, figsize=(12, 3.8))
a[0].hist(expl.image_share, bins=30, color='#55A868'); a[0].set_title('Per-post IMAGE share of the decision'); a[0].set_xlabel('image share (0 = text decides, 1 = image decides)')
sns.boxplot(data=expl.assign(cls=expl.label.map(LABEL_NAMES)), x='cls', y='image_share', hue='correct', ax=a[1])
a[1].set_title('Image share by true class and correctness')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/ablation_per_post.png', dpi=150); plt.show()

In [ ]:
# Show the posts where the IMAGE drove the decision most, and where the TEXT did
from PIL import Image
def show(rows, title):
    fig, axes = plt.subplots(1, len(rows), figsize=(4 * len(rows), 4.6)); fig.suptitle(title)
    for a, r in zip(axes, rows.itertuples()):
        a.imshow(Image.open(f'{IMG_DIR}/{r.id}.jpg')); a.axis('off')
        a.set_title(f'{r.clean_title[:38]}\ntrue {LABEL_NAMES[r.label]} | pred {LABEL_NAMES[r.pred]}\n'
                    f'text {r.text_share:.2f} / image {r.image_share:.2f}', fontsize=8)
    plt.tight_layout(); plt.show()
show(expl[expl.correct].nlargest(4, 'image_share'), 'Correct predictions driven mostly by the IMAGE')
show(expl[expl.correct].nlargest(4, 'text_share'), 'Correct predictions driven mostly by the TEXT')
show(expl[~expl.correct].sample(4, random_state=0), 'Some errors')

## 8 · Save the best model and deploy as a Streamlit app  *(Objective 5)*
The app takes a headline + an image (upload or URL) and shows the verdict, confidence and how much the text vs. image drove it.

In [ ]:
best_model = runs[(BEST_MODEL, best_seed)][0]
torch.save(best_model.state_dict(), f'{MODEL_DIR}/best_model.pt')
hp = {k: v for k, v in HP[BEST_MODEL].items() if k not in ('lr', 'wd')}
cfg = {'model_name': BEST_MODEL, 'model_kwargs': hp, 'seed': int(best_seed), 'max_len': MAX_LEN,
       'text_pool': TEXT_POOL, 'img_size': IMG_SIZE,
       'test_accuracy': float(runs[(BEST_MODEL, best_seed)][2]['accuracy']),
       'test_macro_f1': float(runs[(BEST_MODEL, best_seed)][2]['macro_f1']),
       'mean_test_macro_f1_5_seeds': float(summary.loc[BEST_MODEL, 'f1_mean'])}
json.dump(cfg, open(f'{MODEL_DIR}/config.json', 'w'), indent=2)
print(json.dumps(cfg, indent=2))

In [ ]:
%%writefile {ROOT}/app.py
"""
Streamlit app — Multimodal Fake News Detection (Text + Image fusion)
Needs in the same folder:  fakenews_core.py,  models/best_model.pt,
                           models/config.json,  models/feature_stats.npz
Run:  streamlit run app.py
"""
import os, json
from io import BytesIO
import numpy as np
import torch
import streamlit as st
from PIL import Image

import fakenews_core as core

HERE = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(HERE, "models")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

st.set_page_config(page_title="Fake News Detector", page_icon="🔎", layout="centered")


@st.cache_resource(show_spinner="Loading DistilBERT, ResNet50 and the fusion model…")
def load_everything():
    cfg = json.load(open(os.path.join(MODEL_DIR, "config.json")))
    s = np.load(os.path.join(MODEL_DIR, "feature_stats.npz"))
    model = core.build_model(cfg["model_name"], **cfg["model_kwargs"])
    model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "best_model.pt"), map_location=DEVICE))
    model.to(DEVICE).eval()
    fx = core.FeatureExtractor(device=DEVICE, max_len=cfg["max_len"], text_pool=cfg["text_pool"],
                               img_size=cfg["img_size"])
    return cfg, s, model, fx


def load_image_from_url(url):
    import requests
    r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")


def analyse(headline, image, cfg, s, model, fx):
    t = core.standardise(fx.text([headline]), s["text_mean"], s["text_std"]).astype(np.float32)
    i = core.standardise(fx.image([image]), s["image_mean"], s["image_std"]).astype(np.float32)
    t, i = torch.tensor(t, device=DEVICE), torch.tensor(i, device=DEVICE)
    if len(model.uses) == 2:
        return core.explain(model, t, i)
    p, _ = core.predict_proba(model, t, i)          # single-modality model: no explanation
    return {"pred": p.argmax(1).cpu().numpy(), "prob": p.cpu().numpy()}


# ----------------------------------------------------------------- UI
st.title("🔎 Multimodal Fake News Detector")
st.caption("Judges a post's credibility from its **headline and image together** "
           "(DistilBERT + ResNet50 features, fused by a neural network trained on Fakeddit).")

try:
    cfg, stats, model, fx = load_everything()
except FileNotFoundError as e:
    st.error(f"Model files not found ({e.filename}). Run the notebook's save cell first, "
             "and keep the `models/` folder next to app.py.")
    st.stop()

with st.sidebar:
    st.subheader("Model")
    st.write(f"**{cfg['model_name'].replace('_', ' ').title()}**")
    st.metric("Test macro-F1 (5-seed mean)", f"{cfg['mean_test_macro_f1_5_seeds']:.3f}")
    st.metric("Test accuracy (saved seed)", f"{cfg['test_accuracy']:.3f}")
    st.caption("Research prototype trained on Reddit posts. Use it as a signal, not as a fact-check.")

headline = st.text_area("Headline / post text", placeholder="e.g. Scientists discover a new species of glowing frog")
src = st.radio("Image", ["Upload", "Image URL"], horizontal=True)
image = None
if src == "Upload":
    f = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png", "webp"])
    if f:
        image = Image.open(f).convert("RGB")
else:
    url = st.text_input("Image URL")
    if url:
        try:
            image = load_image_from_url(url)
        except Exception as e:
            st.warning(f"Couldn't load that image: {e}")
if image is not None:
    st.image(image, width=320)

if st.button("Analyse", type="primary", disabled=not (headline.strip() and image is not None)):
    with st.spinner("Analysing…"):
        out = analyse(headline.strip(), image, cfg, stats, model, fx)
    pred, prob = int(out["pred"][0]), out["prob"][0]
    conf = float(prob[pred])
    if pred == 0:
        st.error(f"### Likely FAKE  ·  {conf:.0%} confidence")
    else:
        st.success(f"### Likely REAL  ·  {conf:.0%} confidence")
    st.progress(float(prob[1]), text=f"P(real) = {prob[1]:.2f}   ·   P(fake) = {prob[0]:.2f}")

    if "text_share" in out:
        st.subheader("What drove the decision?")
        ts, ims = float(out["text_share"][0]), float(out["image_share"][0])
        c1, c2 = st.columns(2)
        c1.metric("Text contribution", f"{ts:.0%}")
        c2.metric("Image contribution", f"{ims:.0%}")
        st.bar_chart({"share": {"Text": ts, "Image": ims}})
        pt, pi = out["prob_text_only_view"][0], out["prob_image_only_view"][0]
        st.caption(f"Text alone suggests **{core.LABEL_NAMES[int(pt.argmax())]}** ({pt.max():.0%}); "
                   f"image alone suggests **{core.LABEL_NAMES[int(pi.argmax())]}** ({pi.max():.0%}). "
                   "Contribution = how much the verdict's probability changes when that input is removed.")
        if "gate_text" in out:
            st.caption(f"Attention gate weights — text {out['gate_text'][0]:.2f}, image {out['gate_image'][0]:.2f}")
        if pt.argmax() != pi.argmax():
            st.info("The text and the image point in different directions — a common pattern in "
                    "out-of-context or misleading posts.")

**Run the app on your own laptop instead:** copy `app.py`, `fakenews_core.py` and the `models/` folder from Drive into one folder, then
```
pip install -r requirements.txt
streamlit run app.py
```

## 9 · Notes for the report — limitations
- Features are **frozen**; fine-tuning DistilBERT/ResNet end-to-end (or using CLIP) would likely score higher but needs more GPU time.
- 6k-post subset of an 880k dataset; Fakeddit labels come from subreddit membership (distant supervision), so some labels are noisy.
- Dead image links make the downloadable subset slightly biased toward posts whose images survive.
- Ablation explains *which modality* mattered, not *which words/regions* — Grad-CAM / attention over tokens would be the next step.